# Tool-using fine-tuned parser (ft-t1): train + parse, one size per session (GPU)

Trains a LoRA adapter on data **v3-tools** (`finetune/data/v3_tools/`: data v2 with the fill-with-median and
fill-with-most-common examples as multi-turn tool trajectories; built by `finetune/data_gen/build_tools_data.py`)
with `finetune/train_qlora_tools.py` (loss on every assistant turn), then parses the 120 tasks with tools
(`eval/parse_contracts_tools_ft.py`). Same settings as the current fine-tuning (1 epoch, batch 1 x 16).

Set `SIZE = "1_5b"` or `"3b"` (one per session: ~4 h / ~8 h, estimates). Run `SMOKE = True` first (20 steps).
Outputs: `ft_tools_<size>.zip` (adapter, log, run info) and `parse_ft_tools_<size>.zip`. Unzip the parse zip to
`results/stage1/extensions/intent_ft_tools_<size>/` and run `.venv/bin/python eval/ext_score.py`.
Settings: GPU T4 x1, Internet on.

In [ ]:
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
SIZE = "1_5b"                   # "1_5b" or "3b"
SMOKE = True                    # True: 20 steps (check memory, loss, time per step); False: full training + parsing
BACKEND = "hf"
BASE = {"1_5b": "Qwen/Qwen2.5-1.5B-Instruct", "3b": "Qwen/Qwen2.5-3B-Instruct"}[SIZE]
CONFIG = f"finetune/configs/qwen25_{SIZE}_tools.yaml"
SYSTEM = f"intent_ft_tools_{SIZE}"
NEEDED = ["finetune/train_qlora_tools.py", "finetune/train_qlora.py", "eval/parse_contracts_tools_ft.py", CONFIG,
          "finetune/data_gen/build_tools_data.py"]

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json, collections, zipfile, datetime
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
WORK = "/kaggle/working"
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)
if not os.path.exists("finetune/data/v3_tools/sft_train.jsonl"):        # build it (CPU, ~1 min) if not pushed
    subprocess.run(["python", "finetune/data_gen/build_tools_data.py"], check=True)
print(open("finetune/data/v3_tools/sft_stats.json").read()[:600])

In [ ]:
# ---------------- Libraries: training stack (as the current fine-tuning notebook) + pinned pandas for parsing ----------------
subprocess.run(["pip", "install", "-q", "-r", "finetune/requirements-train.txt"], check=True)
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available()

In [ ]:
def sh(cmd, log=None):
    """Run a step, stream its output live, keep the full log."""
    t0 = time.time()
    p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if "Loading weights" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        open(log, "a").write(f"$ {' '.join(cmd)}\n" + "".join(lines) + f"\n--- exit {rc}, {(time.time() - t0) / 60:.1f} min ---\n")
    assert rc == 0, f"failed: {' '.join(cmd)}"
    return (time.time() - t0) / 60

out = f"finetune/runs/qwen25_{SIZE}_tools" + ("_smoke" if SMOKE else "")
sets = [f"output_dir={out}"] + (["max_steps=20", "eval_steps=10", "logging_steps=2", "limit_val=64"] if SMOKE
                                else ["limit_val=200"])
TRAIN_ENV = dict(os.environ)          # training uses Kaggle's stack, like finetune_qlora_kaggle.ipynb
p = subprocess.Popen(["python", "finetune/train_qlora_tools.py", "--config", CONFIG, "--set", *sets], env=TRAIN_ENV,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
t0 = time.time()
for line in p.stdout:
    print(line, end="", flush=True)
assert p.wait() == 0, "training failed"
mins = (time.time() - t0) / 60
info = json.load(open(f"{out}/run_info.json"))
print(f"trained in {mins:.1f} min | loss {info['first_logged_loss']} -> {info['last_logged_loss']} | prompt {info['parser_prompt_version']}")
if SMOKE:
    print(f"estimated full training: ~{mins * ((info['n_train'] + 15) // 16) / info['steps'] / 60:.1f} h (+ evaluations and parsing)")
open(f"{out}/git_commit.txt", "w").write(COMMIT)
print("saved:", shutil.make_archive(f"{WORK}/ft_tools_{SIZE}" + ("_smoke" if SMOKE else ""), "zip", out))

In [ ]:
# ---------------- Parse the 120 tasks with tools (skipped in SMOKE mode) ----------------
if not SMOKE:
    pout = f"{WORK}/parse_ft_tools_{SIZE}"
    mins = sh(["python", "eval/parse_contracts_tools_ft.py", "--model", BASE, "--load-in-4bit", "--adapter",
               f"{out}/adapter", "--backend", BACKEND, "--system", SYSTEM, "--out", pout, "--cache-dir", f"{WORK}/llm_cache"])
    R = [json.loads(l) for l in open(f"{pout}/parses.jsonl")]
    print(f"parsed in {mins:.1f} min | status {dict(collections.Counter(r['parse_status'] for r in R))} | "
          f"tasks with a tool call {sum(r['n_tool_calls'] > 0 for r in R)}")
    open(f"{pout}/git_commit.txt", "w").write(COMMIT)
    print("saved:", shutil.make_archive(pout, "zip", pout))